## Drivers Data transform Bronze to Silver

In [0]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00-common/01-Environment_Config

In [0]:
%run ../00-common/03-Silver_Helpers

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.drivers"
silver_table = f"{catalog_name}.{silver_schema}.drivers"

In [0]:
from pyspark.sql import functions as f

In [0]:
# To use reader api options 
drivers_df = spark.read.table(bronze_table).filter((f.col("batch_id") == v_batch_id))

# drivers_df = spark.table(bronze_table) - without any reader options
# Both returns the same results

In [0]:
drivers_dropped_df = drivers_df.drop(f.col("url"))

In [0]:
# For column rename we can use ALIAS funcrion or also the below function

column_rename = {
    "driverId" : "driver_id",
    "dateOfBirth" : "date_of_birth"
    }

drivers_renamed_df = drivers_dropped_df.withColumnsRenamed(column_rename)

In [0]:
drivers_concat_df = drivers_renamed_df.withColumn("driver_name", 
                                                  f.initcap(
                                                      f.concat_ws(' ', f.col("name.givenName"), f.col("name.familyName"))))\
                                                          .drop("name")

In [0]:
# To Drop duplicates and Null values in a column - distinct(), dropDuplicates() & filter()

drivers_valid_df = drivers_concat_df.dropDuplicates(["driver_id"]).filter(f.col("driver_id").isNotNull())

In [0]:
drivers_final_df = drivers_valid_df.withColumn("nationality", f.initcap("nationality"))

In [0]:
columns = [
    "driver_name",
    "date_of_birth",
    "nationality",
    "ingestion_timestamp",
    "source_file",
    "batch_id"
]

write_to_silver(drivers_final_df, silver_table, "trg.driver_id = src.driver_id", columns)